In [1]:
import numpy as np
import scipy.signal as signal
import scanpy as sc
from tqdm import tqdm
import pandas as pd
import anndata as ad

In [2]:
def detect_peaks(spectrum, mz_axis, prominence=0.01, intensity_threshold=0.0):
    """Detect peaks with prominence and apply intensity threshold."""
    peaks, properties = signal.find_peaks(spectrum, prominence=prominence)
    intensities = spectrum[peaks]

    # Filter peaks based on intensity threshold
    valid = intensities >= intensity_threshold
    return mz_axis[peaks][valid], intensities[valid]

def adaptive_tolerance(mz, resolution, mode="da"):
    """Calculate m/z tolerance dynamically based on resolution."""
    if mode == "da":
        return mz / resolution
    elif mode == "ppm":
        return 1e6 / resolution
    else:
        raise ValueError("Mode must be 'da' or 'ppm'")

def mz_switch_calculator(ppm_tolerance, da_tolerance):
    return da_tolerance/ppm_tolerance *1e6


def ppm_diff(ref_mz, target_mz):
    return (target_mz - ref_mz) / ref_mz * 1e6


def find_top_peaks_global(adata, prominence=0.01, n_peaks=100, resolution=30000, intensity_threshold=0.0):
    all_peaks = []
    mz_axis = adata.var_names.astype(float).values

    for i in tqdm(range(adata.shape[0]), desc="Detecting peaks"):
        spectrum = adata.X[i].toarray().flatten()
        mz_peaks, intensities = detect_peaks(spectrum, mz_axis, prominence, intensity_threshold)
        all_peaks.extend(list(zip(mz_peaks, intensities)))

    all_peaks = np.array(all_peaks, dtype=[("mz", float), ("intensity", float)])
    sorted_peaks = np.sort(all_peaks, order="intensity")[::-1]

    selected_peaks = []
    for peak in sorted_peaks:
        if len(selected_peaks) >= n_peaks:
            break

        def is_far_enough(p):
            da_tol = adaptive_tolerance(peak["mz"], resolution, mode="da")
            return abs(p["mz"] - peak["mz"]) > da_tol

        if all(is_far_enough(p) for p in selected_peaks):
            selected_peaks.append(peak)

    return np.sort(np.array([p["mz"] for p in selected_peaks]))

def extract_and_aggregate_peaks(adata, selected_mz, resolution=30000):
    mz_axis = adata.var_names.astype(float).values
    X = adata.X

    aggregated_intensities = []
    aggregated_mz_names = []

    for target_mz in selected_mz:
        da_tol = adaptive_tolerance(target_mz, resolution, mode="da")
        diff_array = np.abs(mz_axis - target_mz)
        within_tolerance = np.where(diff_array <= da_tol)[0]

        if len(within_tolerance) == 0:
            summed_intensity = np.zeros((adata.shape[0],))
            avg_mz = target_mz
        else:
            summed = X[:, within_tolerance].sum(axis=1)
            summed_intensity = np.asarray(summed).flatten()
            avg_mz = mz_axis[within_tolerance].mean()

        aggregated_intensities.append(summed_intensity)
        aggregated_mz_names.append(f"{avg_mz:.6f}")

    new_X = np.vstack(aggregated_intensities).T
    new_adata = ad.AnnData(
        X=new_X,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=aggregated_mz_names)
    )

    return new_adata

In [ ]:
# === File Paths ===
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad"
output_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_010_1000_20000_1000.h5ad"

# === Parameters ===
prominence = 0.01
top_n = 1000
intensity_threshold = 1000.0
mz_switch = mz_switch_calculator(ppm_tolerance, da_tolerance)
resolution = 20000  # <- put your instrument's resolution here


# === Load and process ===
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

selected_mz = find_top_peaks_global(
    adata,
    prominence=prominence,
    n_peaks=top_n,
    resolution=resolution,
    intensity_threshold=intensity_threshold
)

adata_top_peaks = extract_and_aggregate_peaks(
    adata,
    selected_mz,
    resolution=resolution
)

adata_top_peaks.write(output_file)
print(f"Reduced AnnData with top {top_n} peaks saved to: {output_file}")

/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a.h5ad


Detecting peaks: 100%|██████████| 16605/16605 [01:14<00:00, 223.98it/s]


Reduced AnnData with top 1000 peaks saved to: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_010_1000_20000_1000.h5ad


In [4]:
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_peaks_010_1000_20000_1000.h5ad"
adata = sc.read_h5ad(input_file)

In [5]:
print(adata)

AnnData object with n_obs × n_vars = 16605 × 820
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status'


In [6]:
adata.var

""
96.926097
112.898041
114.892377
136.060806
137.025085
...
1947.522661
1949.681363
1950.201926
1955.064065


In [7]:
print(adata)

AnnData object with n_obs × n_vars = 16605 × 820
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status'


In [8]:
adata

AnnData object with n_obs × n_vars = 16605 × 820
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status'

In [10]:
import plotly.graph_objects as go
import numpy as np

def plot_top_peaks_stem(adata, title="Top 1000 Peaks by Max Intensity (Stem Plot)"):
    # Get m/z values
    mz_axis = adata.var_names.astype(float).values

    # Get max intensity per m/z (not sum)
    if hasattr(adata.X, "toarray"):  # sparse
        max_intensity = np.asarray(adata.X.max(axis=0)).ravel()
    else:  # dense
        max_intensity = np.max(adata.X, axis=0)

    # Find top 1000 peaks by max intensity
    top_indices = np.argsort(max_intensity)[::-1][:1000]
    mz_top = mz_axis[top_indices]
    intensity_top = max_intensity[top_indices]

    # Sort m/z values for cleaner appearance
    sorted_order = np.argsort(mz_top)
    mz_top = mz_top[sorted_order]
    intensity_top = intensity_top[sorted_order]

    # Create stem plot (vertical lines + markers)
    fig = go.Figure()

    # Vertical lines (stems)
    for x, y in zip(mz_top, intensity_top):
        fig.add_trace(go.Scatter(
            x=[x, x],
            y=[0, y],
            mode='lines',
            line=dict(color='crimson', width=1),
            showlegend=False,
            hoverinfo='skip'
        ))

    # Markers (dots at the top of each stem)
    fig.add_trace(go.Scatter(
        x=mz_top,
        y=intensity_top,
        mode='markers',
        marker=dict(size=4, color='crimson'),
        hovertemplate='Avg m/z: %{x:.4f}<br>Max Intensity: %{y:.2f}',
        name='Max Intensity'
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Averaged m/z",
        yaxis_title="Max Intensity (across sample)",
        template="plotly_white",
        dragmode="zoom",
        hovermode="closest"
    )

    fig.show()


In [11]:
plot_top_peaks_stem(adata)

In [12]:
import plotly.graph_objects as go
import numpy as np

def plot_top_peaks_by_max_intensity(adata, title="Top 1000 Peaks by Max Intensity"):
    # Get m/z values
    mz_axis = adata.var_names.astype(float).values

    # Get max intensity per m/z (not sum)
    if hasattr(adata.X, "toarray"):  # sparse
        max_intensity = np.asarray(adata.X.max(axis=0)).ravel()
    else:  # dense
        max_intensity = np.max(adata.X, axis=0)

    # Find top 1000 peaks by max intensity
    top_indices = np.argsort(max_intensity)[::-1][:1000]
    mz_top = mz_axis[top_indices]
    intensity_top = max_intensity[top_indices]

    # Sort m/z values for nicer plot appearance
    sorted_order = np.argsort(mz_top)
    mz_top = mz_top[sorted_order]
    intensity_top = intensity_top[sorted_order]

    # Plot using Plotly
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=mz_top,
        y=intensity_top,
        mode='lines+markers',
        marker=dict(size=4, color='crimson'),
        line=dict(width=1),
        hovertemplate='Avg m/z: %{x:.4f}<br>Max Intensity: %{y:.2f}',
        name='Max Intensity'
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Averaged m/z",
        yaxis_title="Max Intensity (across sample)",
        template="plotly_white",
        dragmode="zoom",
        hovermode="closest"
    )

    fig.show()

In [13]:
plot_top_peaks_by_max_intensity(adata)
